# Predicting Irrigation Need

Link to Competittion: https://www.kaggle.com/competitions/playground-series-s6e4/overview

## Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.figsize'] = (12, 6)

import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, accuracy_score, confusion_matrix, mean_squared_error, classification_report, balanced_accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold, StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.cluster import KMeans
from sklearn.utils.class_weight import compute_sample_weight

from common import *

In [2]:
from platform import python_version
print('python: ', python_version())
print('pandas: ', pd.__version__)
print('numpy: ', np.__version__)
import matplotlib
print('matplotlib: ', matplotlib.__version__)
print('seaborn: ', sns.__version__)
import sklearn
print('sklearn: ', sklearn.__version__)
print('xgboost: ', xgb.__version__)

python:  3.13.11
pandas:  2.3.3
numpy:  2.3.5
matplotlib:  3.10.8
seaborn:  0.13.2
sklearn:  1.8.0
xgboost:  3.2.0


## Helpers

## Load data

In [3]:
train_df = pd.read_csv('archive/train.csv')
test_df = pd.read_csv('archive/test.csv')
orig_df = pd.read_csv('archive/irrigation_prediction.csv')

## Call the pipeline

In [4]:
df = (train_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

## Concat original dataset

In [5]:
orig_df_cleaned = (orig_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [6]:
df = pd.concat([df, orig_df_cleaned])

## Features

In [7]:
target = get_target()

In [8]:
features = get_features(df)

In [9]:
features

['soil_type',
 'soil_ph',
 'soil_moisture',
 'organic_carbon',
 'electrical_conductivity',
 'temperature_c',
 'humidity',
 'rainfall_mm',
 'sunlight_hours',
 'wind_speed_kmh',
 'crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'field_area_hectare',
 'mulching_used',
 'previous_irrigation_mm',
 'region']

In [10]:
categorical_features = [
    'crop_type',
    'crop_growth_stage',
    'season',
    'irrigation_type',
    'water_source',
    'soil_type',
    'region'
]

In [11]:
numerical_features = [f for f in features if f not in categorical_features]

In [12]:
categorical_features

['crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'soil_type',
 'region']

In [13]:
numerical_features

['soil_ph',
 'soil_moisture',
 'organic_carbon',
 'electrical_conductivity',
 'temperature_c',
 'humidity',
 'rainfall_mm',
 'sunlight_hours',
 'wind_speed_kmh',
 'field_area_hectare',
 'mulching_used',
 'previous_irrigation_mm']

In [14]:
df['mulching_used'] = df['mulching_used'].map({'Yes': 1, 'No': 0})

In [15]:
df[features]

,soil_type,soil_ph,soil_moisture,organic_carbon,electrical_conductivity,temperature_c,humidity,rainfall_mm,sunlight_hours,wind_speed_kmh,crop_type,crop_growth_stage,season,irrigation_type,water_source,field_area_hectare,mulching_used,previous_irrigation_mm,region
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,0,112.16,East
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,1,47.16,South
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,1,110.38,North
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,1,53.85,South
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,0,93.19,South
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Silt,7.01,26.67,0.86,0.76,27.61,52.20,1075.12,7.41,19.66,Sugarcane,Sowing,Kharif,Drip,Groundwater,2.62,1,92.44,South
9996,Clay,5.40,49.44,0.90,1.19,34.03,52.31,1591.84,9.86,5.66,Maize,Sowing,Kharif,Rainfed,Groundwater,4.87,0,15.46,South
9997,Loamy,4.97,60.63,0.99,1.30,36.68,68.16,2384.87,10.75,13.40,Potato,Harvest,Kharif,Canal,Groundwater,10.08,1,116.36,North
9998,Loamy,7.12,44.33,1.56,1.08,31.50,64.83,2397.01,4.03,3.05,Sugarcane,Harvest,Kharif,Rainfed,Reservoir,11.11,1,118.17,East


In [16]:
df[target] = df[target].map({'Low': 0, 'Medium': 1, 'High': 2})

In [17]:
df[categorical_features] = df[categorical_features].astype('category')

In [18]:
categorical_features

['crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'soil_type',
 'region']

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 640000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   soil_type                640000 non-null  category
 1   soil_ph                  640000 non-null  float64 
 2   soil_moisture            640000 non-null  float64 
 3   organic_carbon           640000 non-null  float64 
 4   electrical_conductivity  640000 non-null  float64 
 5   temperature_c            640000 non-null  float64 
 6   humidity                 640000 non-null  float64 
 7   rainfall_mm              640000 non-null  float64 
 8   sunlight_hours           640000 non-null  float64 
 9   wind_speed_kmh           640000 non-null  float64 
 10  crop_type                640000 non-null  category
 11  crop_growth_stage        640000 non-null  category
 12  season                   640000 non-null  category
 13  irrigation_type          640000 non-null  category


## Stratified KFold Loop

In [20]:
xgb_params = {
    'n_jobs': -1,
    'enable_categorical': True,
    'random_state': 123
}

In [21]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)
scores = []

X, y = df[features], df[target]

In [22]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    fold_sample_weights = compute_sample_weight('balanced', y_train)
    model.fit(X_train, y_train, sample_weight=fold_sample_weights)

    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    scores.append(score)
    print(f"Fold {fold}:  {score:.5f}")

print(f"Mean: {np.mean(scores):.5f} ± {np.std(scores):.5f}")

Fold 0:  0.97123
Fold 1:  0.96537
Fold 2:  0.97006
Fold 3:  0.96944
Fold 4:  0.96975
Fold 5:  0.97080
Fold 6:  0.97211
Fold 7:  0.97375
Fold 8:  0.97150
Fold 9:  0.96788
Mean: 0.97019 ± 0.00221


## Final Model

In [23]:
xgb_final_model = XGBClassifier(**xgb_params)
final_sample_weights = compute_sample_weight('balanced', df[target])
xgb_final_model.fit(df[features], df[target], sample_weight=final_sample_weights)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [24]:
df = (test_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [25]:
df['mulching_used'] = df['mulching_used'].map({'Yes': 1, 'No': 0})

In [26]:
df[categorical_features] = df[categorical_features].astype('category')

In [27]:
df

,soil_type,soil_ph,soil_moisture,organic_carbon,electrical_conductivity,temperature_c,humidity,rainfall_mm,sunlight_hours,wind_speed_kmh,crop_type,crop_growth_stage,season,irrigation_type,water_source,field_area_hectare,mulching_used,previous_irrigation_mm,region
0,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,1,47.48,West
1,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,1,56.43,South
2,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,1,20.00,East
3,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,0,102.99,North
4,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,1,13.33,Central
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269995,Sandy,5.63,51.90,0.68,2.58,33.27,72.09,2326.61,7.09,10.02,Potato,Vegetative,Rabi,Rainfed,River,2.93,1,43.49,East
269996,Loamy,7.84,45.16,0.85,1.04,27.55,45.16,2322.37,5.15,5.62,Wheat,Vegetative,Rabi,Canal,Groundwater,11.23,1,92.03,West
269997,Loamy,7.83,11.02,1.56,1.90,23.39,64.87,996.72,10.44,9.98,Maize,Vegetative,Zaid,Sprinkler,Groundwater,2.88,1,34.02,East
269998,Silt,7.12,10.18,1.32,2.65,41.09,58.04,1130.71,5.11,1.46,Rice,Harvest,Kharif,Rainfed,River,5.71,1,3.92,East


In [28]:
preds = xgb_final_model.predict(df[features])

In [29]:
preds

array([0, 0, 0, ..., 1, 0, 1], shape=(270000,))

In [30]:
submission_df = pd.DataFrame(test_df['id'].copy())

In [31]:
submission_df['Irrigation_Need'] = preds

In [32]:
submission_df

,id,Irrigation_Need
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0
...,...,...
269995,899995,1
269996,899996,0
269997,899997,1
269998,899998,0


In [33]:
submission_df['Irrigation_Need'] = submission_df['Irrigation_Need'].map({0: 'Low', 1: 'Medium', 2: 'High'})

In [34]:
submission_df['Irrigation_Need'].value_counts()

Irrigation_Need
Low       159865
Medium    100728
High        9407
Name: count, dtype: int64

In [35]:
last_submission = pd.read_csv(find_last_submission_file())

In [36]:
last_submission['Irrigation_Need'].value_counts()

Irrigation_Need
Low       159865
Medium    100728
High        9407
Name: count, dtype: int64

In [37]:
# if np.allclose(last_submission['Irrigation_Need'], submission_df['Irrigation_Need']):
if all(last_submission['Irrigation_Need'].value_counts() == submission_df['Irrigation_Need'].value_counts()):
    # they are the same, don't same
    print('skipping save')
else:
    submission_df.to_csv(find_next_submission_file(), index=False)
    print('saving file')

skipping save
